In [ ]:
# Summary:

# Developing a semi-supervised learning model on Diabetes dataset 
# using a super learner and deploying the model on other datasets.

# Step 1: Data pre-processing phase

# Remove outliers from all columns.
# Impute missing values in all columns.
# Normalize all columns.

# Step 2: Unsupervised Learning for generating labels

# Use K-means clustering on three features of Glucose, 
# BMI and Age to cluster data into two clusters.
# Assign ‘Diabetes’ name to the cluster with higher average Glucose 
# and ‘No Diabetes’ to the other cluster.
# Add a new column (Outcome) to the dataset containing 1 for ‘Diabetes’ 
# and 0 for ‘No Diabetes’. Use these values as labels for classification (step 4).

# Step 3: Feature Extraction

# Split data into test and training sets (consider 20% for test).
# Use PCA on the training data to create 3 new components 
# from existing features (all columns except outcome).
# Transfer training and test data to the new dimensions (PCs).

# Step 4: Classification using a super learner

# Define three classification models as base classifiers 
# consisting of Naïve Bayes, Neural Network, and KNN.
# Define a decision tree as the meta learner.
# Train decision tree (meta learner) on outputs of three base classifiers 
# using 5-fold cross validation.
# Find hyperparameters for all these models which provide the best accuracy rate.
# Report accuracy of the model on the test data.

# Step 5: Employing the model on other datasets

# Use the last column of the assigned dataset as outcome (label).
# Use your current code for steps 1,3, and 4 
# with minor changes (e.g., encoding categorical variables) 
# to train your model on the new dataset and calculate the accuracy.
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
data = pd.read_csv('Datasets/Airline_Satisfaction.csv')
# print(data.head(5))
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Gender                 103904 non-null  object 
 1   CustomerType           103904 non-null  object 
 2   Age                    103904 non-null  int64  
 3    TravelType            103904 non-null  object 
 4   Class                  103904 non-null  object 
 5   FlightDistance         103904 non-null  int64  
 6   InflightWifi           103904 non-null  int64  
 7   TimeConvenient         103904 non-null  int64  
 8   OnlineBooking          103904 non-null  int64  
 9   GateLocation           103904 non-null  int64  
 10  FoodDrink              103904 non-null  int64  
 11  OnlineBoarding         103904 non-null  int64  
 12  SeatComfort            103904 non-null  int64  
 13  InflightEntertainment  103904 non-null  int64  
 14  OnboardService         103904 non-nu

: 

In [ ]:
#pre processing
target = 'satisfied'
y_new = data[target]
X_new = data.drop(columns=[target])
# these two features are categorical and need to be encoded for one-hot encoding
categorical_features = ['trt', 'strat'] 
X_new_encoded = pd.get_dummies(X_new, columns=categorical_features, drop_first=True)

# Identify binary columns and dummy columns
binary_cols = ['hemo', 'homo', 'drugs', 'oprior', 'z30', 'race', 'gender', 
               'str2', 'symptom', 'treat', 'offtrt']
dummy_cols = [col for col in X_new_encoded.columns if col not in X_new.columns]
# add binary columns and dummy columns
all_binary_cols = binary_cols + dummy_cols

continuous_features = [
    col for col in X_new_encoded.columns 
    if col not in all_binary_cols
]
data_processed = X_new_encoded.copy()

for col in continuous_features:
    Q1 = data_processed[col].quantile(0.25)
    Q3 = data_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)
    data_processed.loc[
        (data_processed[col] < lower_bound) | 
        (data_processed[col] > upper_bound),
        col
    ] = np.nan
imputer = SimpleImputer(strategy='median')
data_processed[continuous_features] = imputer.fit_transform(data_processed[continuous_features])
print(data_processed[continuous_features].isnull().sum().head())
